In [3]:
import pandas as pd
from pathlib import Path
from datetime import datetime
import re
from collections import Counter, defaultdict


In [4]:
base_path = Path('c:/App/Mapping_oLD')
users_path = base_path / 'Users.csv'
vehicles_path = base_path / 'Vehicles.csv'
trips_path = base_path / 'Trips.csv'

users_df = pd.read_csv(users_path, dtype=str)
vehicles_df = pd.read_csv(vehicles_path, dtype=str)
trips_df = pd.read_csv(trips_path, dtype=str)

print('Users:', users_df.shape)
print('Vehicles:', vehicles_df.shape)
print('Trips:', trips_df.shape)

users_df.head(5), vehicles_df.head(5), trips_df.head(5)


Users: (64, 50)
Vehicles: (17, 4)
Trips: (64, 50)


(                Name User Group  STATUS Location Setup  MON  TUE  WED THUR  \
 0     Muzill Izmarai        EMT  ACTIVE     SACRAMENTO  YES  YES  YES  YES   
 1         Ghazi Abed        EMT  ACTIVE     SACRAMENTO   NO   NO  YES  YES   
 2      Evan  Seidell    Non-EMT  ACTIVE     SACRAMENTO  YES  YES  YES  YES   
 3       Meontae Sapp    Non-EMT  ACTIVE    SANTA CLARA  YES  YES  YES  YES   
 4  William Rodriguez    Non-EMT  ACTIVE    SANTA CLARA  YES  YES  YES  YES   
 
    FRI  SAT  ... Unnamed: 40 Unnamed: 41 Unnamed: 42 Unnamed: 43 Unnamed: 44  \
 0  YES  YES  ...         NaN         NaN         NaN         NaN         NaN   
 1   NO  YES  ...         NaN         NaN         NaN         NaN         NaN   
 2  YES  YES  ...         NaN         NaN         NaN         NaN         NaN   
 3  YES  YES  ...         NaN         NaN         NaN         NaN         NaN   
 4  YES  YES  ...         NaN         NaN         NaN         NaN         NaN   
 
   Unnamed: 45 Unnamed: 46 Unnamed: 

In [5]:
# Helper functions translated from Mapping.html logic

LOS_RULES = {
    'WHEELCHAIR': {'crew':1,'ne':1,'emt':0,'rn':0,'bariatric':False,'sacOnly':False,'vehicles':['Wheelchair Van','Gurney Van']},
    'ELECTRIC WC': {'crew':1,'ne':1,'emt':0,'rn':0,'bariatric':False,'sacOnly':False,'vehicles':['Wheelchair Van','Gurney Van']},
    'BARIATRIC WC': {'crew':1,'ne':1,'emt':0,'rn':0,'bariatric':True,'sacOnly':False,'vehicles':['Gurney Van']},
    'BARIATRIC WHEELCHAIR': {'crew':1,'ne':1,'emt':0,'rn':0,'bariatric':True,'sacOnly':False,'vehicles':['Gurney Van']},
    'GURNEY': {'crew':2,'ne':2,'emt':0,'rn':0,'bariatric':False,'sacOnly':False,'vehicles':['Gurney Van','Ambulance']},
    'BARIATRIC GURNEY': {'crew':2,'ne':2,'emt':0,'rn':0,'bariatric':False,'sacOnly':False,'vehicles':['Gurney Van','Ambulance']},
    'BLS': {'crew':2,'ne':0,'emt':2,'rn':0,'bariatric':False,'sacOnly':True,'vehicles':['Ambulance']},
    'CCT': {'crew':3,'ne':0,'emt':2,'rn':1,'bariatric':False,'sacOnly':True,'vehicles':['Ambulance']},
}
LOS_ALIAS = {'WC':'WHEELCHAIR','WHEELCHAIR':'WHEELCHAIR','ELECTRIC WC':'ELECTRIC WC','EWC':'ELECTRIC WC','BARIATRIC WC':'BARIATRIC WC','BARIATRIC WHEELCHAIR':'BARIATRIC WC','GURNEY':'GURNEY','BARIATRIC GURNEY':'BARIATRIC GURNEY','BLS':'BLS','CCT':'CCT','ALS':'CCT'}
BASE_CAPABILITIES = {
    'SACRAMENTO':['WHEELCHAIR','ELECTRIC WC','BARIATRIC WC','GURNEY','BARIATRIC GURNEY','BLS','CCT'],
    'SANTA CLARA':['WHEELCHAIR','ELECTRIC WC','BARIATRIC WC','GURNEY','BARIATRIC GURNEY','BLS'],
    'SAN JOSE':['WHEELCHAIR','ELECTRIC WC','BARIATRIC WC','GURNEY','BARIATRIC GURNEY','BLS'],
}
NO_EMT_FALLBACK = ['WHEELCHAIR','ELECTRIC WC','BARIATRIC WC']
COMPLETION_MINUTES=45
WR_BLOCK=180
SHIFT_TOTAL=8*60+30
PREP_MAX=60
DAYS=['MON','TUE','WED','THU','FRI','SAT','SUN']


def norm_los(value):
    u = str(value or '').strip().upper()
    if 'ALS' in u:
        return 'CCT'
    return LOS_ALIAS.get(u,u)


def normalize_zone(value):
    if not value:
        return ''
    z = str(value).strip().upper().replace('.', '').replace(',', ' ')
    if 'SAN JOSE' in z or z == 'SJ' or z == 'SJC' or z.startswith('SJ '):
        return 'SANTA CLARA'
    if 'SANTA CLARA' in z or z == 'SC' or z.startswith('SC '):
        return 'SANTA CLARA'
    if 'SACRAMENTO' in z or z == 'SAC' or z.startswith('SAC '):
        return 'SACRAMENTO'
    return z


def normalize_crew_type(value):
    if not value:
        return 'Non-EMT'
    t = str(value).strip().upper()
    if 'RN' in t or 'NURSE' in t:
        return 'RN'
    if 'EMT' in t or 'PARAMEDIC' in t or 'MEDIC' in t:
        return 'EMT'
    return 'Non-EMT'


def normalize_status(value):
    if value is None or str(value).strip() == '':
        return 'Active'
    s = str(value).strip().lower()
    if re.match(r'^(active|working|available|yes|y|on|ok|true)$', s, re.I):
        return 'Active'
    if re.match(r'^(inactive|off|no|n|absent|sick|leave|vacation|vacant|unavailable)$', s, re.I):
        return 'Inactive'
    return 'Active'


def parse_day_flag(value):
    if value is None:
        return 0
    if isinstance(value, bool):
        return 1 if value else 0
    if isinstance(value, (int, float)):
        return 1 if value != 0 else 0
    s = str(value).strip().lower()
    if not s:
        return 0
    if re.match(r'^(1|yes|y|true|x|on)$', s, re.I):
        return 1
    if re.match(r'^(0|no|n|false|off)$', s, re.I):
        return 0
    if re.match(r'^(mon|monday|m)$', s, re.I):
        return 1
    if re.match(r'^(tue|tues|tuesday|t)$', s, re.I):
        return 1
    if re.match(r'^(wed|wednesday|w)$', s, re.I):
        return 1
    if re.match(r'^(thu|thur|thurs|thursday)$', s, re.I):
        return 1
    if re.match(r'^(fri|friday|f)$', s, re.I):
        return 1
    if re.match(r'^(sat|saturday|sa)$', s, re.I):
        return 1
    if re.match(r'^(sun|sunday|su)$', s, re.I):
        return 1
    if re.match(r'^[01]{7}$', s):
        return 1 if s[0] == '1' else 0
    return 0


def parse_days_from_row(row):
    columns = list(row.index)
    out = [0]*7
    day_groups = [
        ['Mon','Monday','MON','MONDAY'],
        ['Tue','Tues','Tuesday','TUE','TUES','TUESDAY'],
        ['Wed','Wednesday','WED','WEDNESDAY'],
        ['Thu','Thur','Thurs','Thursday','THU','THUR','THURS','THURSDAY'],
        ['Fri','Friday','FRI','FRIDAY'],
        ['Sat','Saturday','SAT','SATURDAY'],
        ['Sun','Sunday','SUN','SUNDAY'],
    ]
    for i, keys in enumerate(day_groups):
        for key in keys:
            if key in row and str(row[key]).strip() != '':
                out[i] = parse_day_flag(row[key])
                break
    if any(d != 0 for d in out):
        return out
    working_days = None
    for key in ['Working Days','Work Days','Days','Availability','Available Days']:
        if key in row and str(row[key]).strip() != '':
            working_days = str(row[key]).strip()
            break
    if working_days:
        tokens = re.split(r'[;,|\s]+', working_days)
        dayIndex = {'mon':0,'monday':0,'m':0,'tue':1,'tues':1,'tuesday':1,'t':1,'wed':2,'wednesday':2,'w':2,'thu':3,'thur':3,'thurs':3,'thursday':3,'fri':4,'friday':4,'f':4,'sat':5,'saturday':5,'sa':5,'sun':6,'sunday':6,'su':6}
        for tok in tokens:
            token = tok.strip().rstrip('.').lower()
            if token == 'weekdays':
                for j in range(5):
                    out[j] = 1
                continue
            if token == 'weekends':
                out[5] = 1
                out[6] = 1
                continue
            if '-' in token:
                parts = [p.strip() for p in token.split('-') if p.strip()]
                if len(parts) == 2 and parts[0] in dayIndex and parts[1] in dayIndex:
                    start = dayIndex[parts[0]]
                    end = dayIndex[parts[1]]
                    i = start
                    while True:
                        out[i] = 1
                        if i == end:
                            break
                        i = (i + 1) % 7
                    continue
            if token in dayIndex:
                out[dayIndex[token]] = 1
    if any(out):
        return out
    return [1,1,1,1,1,1,1]


def to_mins(time_str):
    if not time_str or pd.isna(time_str):
        return 0
    t = str(time_str).strip()
    if not t:
        return 0
    parts = t.split(':')
    if len(parts) == 1:
        try:
            return int(parts[0]) * 60
        except ValueError:
            return 0
    try:
        h = int(parts[0])
        m = int(parts[1])
        return h*60 + m
    except ValueError:
        return 0


def normalise_time(value):
    if pd.isna(value) or str(value).strip() == '':
        return ''
    v = str(value).strip()
    v = v.replace('.', ':').replace(' ', '')
    if ':' in v:
        parts = v.split(':')
        if len(parts) == 2 and parts[0].isdigit() and parts[1].isdigit():
            h = int(parts[0])
            m = int(parts[1])
            return f"{h:02d}:{m:02d}"
    if v.isdigit():
        if len(v) <= 2:
            return f"{int(v):02d}:00"
        if len(v) == 3:
            return f"0{v[0]}:{v[1:]}"
        if len(v) == 4:
            return f"{v[:2]}:{v[2:]}"
    return v


def parse_dob(value):
    if pd.isna(value) or str(value).strip() == '':
        return ''
    try:
        return datetime.strptime(str(value).strip(), '%m/%d/%Y').strftime('%m/%d/%Y')
    except Exception:
        try:
            return datetime.strptime(str(value).strip(), '%Y-%m-%d').strftime('%m/%d/%Y')
        except Exception:
            return str(value).strip()


def clean_name(value):
    if pd.isna(value) or str(value).strip() == '':
        return ''
    return re.sub(r'\s+', ' ', str(value).strip()).upper()


def parse_boolean(value):
    if pd.isna(value):
        return False
    s = str(value).strip().lower()
    return s in {'yes','y','true','1'}


def excel_date_to_str(value):
    if pd.isna(value):
        return ''
    s = str(value).strip()
    try:
        dt = datetime.strptime(s, '%m/%d/%Y')
        return dt.strftime('%m/%d/%Y')
    except Exception:
        pass
    try:
        return datetime.strptime(s, '%Y-%m-%d').strftime('%m/%d/%Y')
    except Exception:
        pass
    try:
        # Excel numeric date
        x = float(s)
        base = datetime(1899, 12, 30)
        dt = base + pd.Timedelta(days=x)
        return dt.strftime('%m/%d/%Y')
    except Exception:
        return s


def check_location_and_capability(zone, los):
    return los in BASE_CAPABILITIES.get(zone, [])


def calc_shift_start(pm, travel):
    if travel is None:
        return pm - PREP_MAX
    return pm - PREP_MAX if travel <= 30 else pm - (30 + travel)


def calc_lunch_start(ss):
    return ss + 4*60 + 30


def get_base_to_pickup_mins(trip, travel_cache):
    if not trip.get('zone') or not trip.get('pickupAddr'):
        return None
    key = f"{trip['zone']}|{trip['pickupAddr']}"
    return travel_cache.get(key)


def fallback_block(trip):
    if trip.get('isWR') and trip.get('isWrMaster'):
        return WR_BLOCK
    return COMPLETION_MINUTES + 15


def get_wr_reservation_end(trip, blk, group_keys, group_master_auths, group_end_times):
    if not trip.get('isWR') or not trip.get('patientId') or trip.get('patientId') not in group_keys:
        return trip['pickupMins'] + blk
    if trip['auth'] in group_master_auths and group_end_times.get(trip['patientId']) is not None:
        return group_end_times[trip['patientId']]
    return trip['pickupMins'] + blk


def get_wr_block(trip, group_keys, group_master_auths, group_end_times):
    blk = fallback_block(trip)
    if not trip.get('isWR') or not trip.get('patientId') or trip.get('patientId') not in group_keys:
        return blk
    reservation_end = get_wr_reservation_end(trip, blk, group_keys, group_master_auths, group_end_times)
    return max(0, reservation_end - trip['pickupMins'])


def can_fallback_to_emt(trip):
    if trip['los'] in NO_EMT_FALLBACK:
        return False
    if trip['zone'] == 'SANTA CLARA' and trip['los'] in {'GURNEY', 'BARIATRIC GURNEY'}:
        return False
    return True


def use_emt_for_trip(trip, rule):
    if rule['emt'] > 0:
        return True
    return trip['zone'] == 'SACRAMENTO' and trip['los'] in {'GURNEY', 'BARIATRIC GURNEY'}


def can_do_trip(e, trip, blk, ignore_group_reserved_until=False, ignore_lunch=False, ignore_avail_from=False):
    if not e['active']:
        return False
    pm = trip['pickupMins']
    if not ignore_group_reserved_until and e.get('groupReservedUntil') and pm < e['groupReservedUntil']:
        return False
    if not ignore_avail_from and pm < e.get('availFrom', 0):
        return False
    shift_start = e['shiftStart'] if e['shiftStart'] is not None else calc_shift_start(pm, trip.get('travelMins'))
    shift_end = shift_start + SHIFT_TOTAL
    lunch_start = e['lunchStart'] if e['shiftStart'] is not None else calc_lunch_start(shift_start)
    lunch_end = lunch_start + 30
    if pm < shift_start or pm >= shift_end:
        return False
    if pm + blk > shift_end:
        return False
    if not ignore_lunch and pm < lunch_end and pm + blk > lunch_start:
        return False
    return True


def vehicle_priority(v, los):
    t = str(v['type'] or '').strip().lower().replace('\s+', ' ')
    is_wc = t == 'wheelchair van' or str(v['name']).upper().startswith('WC')
    is_gw = t == 'gurney van' or str(v['name']).upper().startswith('GW')
    is_amb = t == 'ambulance' or str(v['name']).upper().startswith('M')
    if los == 'WHEELCHAIR':
        if is_wc: return 1
        if is_gw: return 2
        return 3
    if los in {'GURNEY', 'BARIATRIC GURNEY'}:
        if is_amb: return 1
        if is_gw: return 2
        return 3
    if los == 'BARIATRIC WC':
        if is_gw: return 1
        return 3
    if los in {'BLS','CCT'}:
        if is_amb: return 1
        return 2
    return 4


def is_veh_free_for_shift(vehicle_name, shift_start, shift_end, owner, veh_shift_map):
    for w in veh_shift_map.get(vehicle_name, []):
        if w['owner'] != owner and shift_start < w['shiftEnd'] and shift_end > w['shiftStart']:
            return False
    return True


def claim_veh_for_shift(vehicle_name, shift_start, shift_end, owner, veh_shift_map):
    veh_shift_map.setdefault(vehicle_name, []).append({'shiftStart': shift_start, 'shiftEnd': shift_end, 'owner': owner})


def assign_vehicle_for_crew(los, zone, bariatric_req, block_start, block_end, owner, vehicles, veh_shift_map):
    allowed_types = [x.lower() for x in LOS_RULES[los]['vehicles']]
    ss = block_start
    se = block_end
    candidates = []
    for v in vehicles:
        if v['location'] != zone:
            continue
        if str(v['status']).lower() not in {'in-service', 'active', 'in service'}:
            continue
        norm_type = str(v['type']).strip().lower()
        if norm_type not in allowed_types:
            continue
        if bariatric_req and not v.get('bariatric', False):
            continue
        if not is_veh_free_for_shift(v['name'], ss, se, owner, veh_shift_map):
            continue
        candidates.append(v)
    candidates.sort(key=lambda a: (vehicle_priority(a, los), a['name']))
    if not candidates:
        return {'vehicle': None}
    v = candidates[0]
    claim_veh_for_shift(v['name'], ss, se, owner, veh_shift_map)
    return {'vehicle': v}


def get_crew_vehicle(e, trip, blk, rule, vehicles, veh_shift_map):
    pm = trip['pickupMins']
    ss = pm
    se = pm + blk
    if e.get('vehicle'):
        if is_veh_free_for_shift(e['vehicle'], ss, se, e['id'], veh_shift_map):
            claim_veh_for_shift(e['vehicle'], ss, se, e['id'], veh_shift_map)
            return e['vehicle']
        return None
    v_info = assign_vehicle_for_crew(trip['los'], trip['zone'], rule.get('bariatric', False), ss, se, e['id'], vehicles, veh_shift_map)
    e['vehicle'] = v_info['vehicle']['name'] if v_info['vehicle'] else None
    return e['vehicle']


<>:326: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<>:326: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
C:\Users\Optiplex\AppData\Local\Temp\ipykernel_23160\745868970.py:326: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
  t = str(v['type'] or '').strip().lower().replace('\s+', ' ')


In [6]:
# Normalize inputs and build records

users = []
for _, row in users_df.iterrows():
    name = str(row.get('Name', '') or row.get('name', '') or row.get('Full Name', '') or row.get('FullName', '') or row.get('Crew Name', '') or row.get('Employee', '') or row.get('Employee Name', '') or '').strip()
    if not name:
        continue
    zone = normalize_zone(row.get('Zone', '') or row.get('zone', '') or row.get('Location Setup', '') or row.get('Location', '') or row.get('Base', '') or row.get('Work Zone', '') or row.get('Region', '') or row.get('Work Location', ''))
    if not zone:
        zone = normalize_zone(row.get('Location Setup', ''))
    users.append({
        'name': name,
        'type': normalize_crew_type(row.get('User Group', '') or row.get('Type', '') or row.get('Role', '') or row.get('Role Name', '') or row.get('Position', '') or row.get('Job', '')),
        'status': normalize_status(row.get('STATUS', '') or row.get('status', '') or row.get('Employment Status', '') or row.get('Active', '') or row.get('Availability', '')),
        'zone': zone,
        'days': parse_days_from_row(row),
    })

vehicles = []
for _, row in vehicles_df.iterrows():
    name = str(row.get('Vehicle Name', '') or row.get('Name', '') or row.get('name', '')).strip()
    if not name:
        continue
    vtype = str(row.get('Vehicle Type', '') or row.get('Type', '') or row.get('type', '')).strip()
    if not vtype:
        if name.startswith('GW'):
            vtype = 'Gurney Van'
        elif name.startswith('WC'):
            vtype = 'Wheelchair Van'
        elif name.startswith('M'):
            vtype = 'Ambulance'
        else:
            vtype = 'Unknown'
    loc = normalize_zone(row.get('Location', '') or row.get('Zone', '') or row.get('Base', '') or row.get('location', ''))
    bariatric = parse_boolean(row.get('Bariatric Capable', '') or row.get('Bariatric', '') or row.get('bariatric', ''))
    vehicles.append({
        'name': name,
        'type': vtype,
        'status': str(row.get('Status', '') or row.get('Vehicle Status', '') or row.get('status', '') or 'In-Service').strip(),
        'location': loc,
        'bariatric': bariatric,
    })

trips = []
for _, row in trips_df.iterrows():
    date_str = excel_date_to_str(str(row.get('Pickup Date', '') or row.get('pickup_date', '') or row.get('PickupDate', '') or row.get('Trip Date', '') or row.get('trip_date', '') or row.get('Date', '')))
    pickup = normalise_time(row.get('Pickup Time', '') or row.get('Pickup time', '') or row.get('pickup', '') or row.get('Time', '') or row.get('Start Time', '') or row.get('PickupTime', ''))
    if not date_str or not pickup:
        continue
    los = norm_los(row.get('Level Of Service', '') or row.get('LOS', '') or row.get('los', '') or row.get('Service Type', '') or row.get('Service', '') or row.get('Case Type', '') or row.get('Transport Type', '') or row.get('Trip Type', ''))
    zone = normalize_zone(row.get('Zone', '') or row.get('zone', '') or row.get('Base', '') or row.get('Region', '') or row.get('Location', '') or row.get('Trip Zone', '') or row.get('Service Area', '') or row.get('Area', ''))
    auth = str(row.get('Auth No', '') or row.get('auth', '') or row.get('Auth', '') or row.get('Authorization', '') or row.get('Order No', '') or row.get('Order Number', '') or row.get('Trip No', '') or row.get('Trip Number', '')).strip()
    wr = str(row.get('Wait & Return', '') or row.get('WR', '') or row.get('Wait Return', '') or row.get('WaitReturn', '') or row.get('Wait and Return', '')).strip().upper()
    specEq = str(row.get('Special Equipment', '') or row.get('Special_Equipment', '') or row.get('special_equipment', '') or row.get('Equipment', '') or row.get('Equip', '')).strip()
    patient = clean_name(row.get('Patient Information', '') or row.get('Patient', '') or row.get('patient_name', '') or row.get('Patient Name', '') or row.get('PatientName', '') or row.get('Client', '') or row.get('Client Name', ''))
    dob = parse_dob(row.get('DOB', '') or row.get('Date of Birth', '') or row.get('Birth Date', '') or row.get('Patient DOB', '') or row.get('Date of Birth', '') or row.get('Patient_DOB', '') or row.get('dob', ''))
    travel_mins = None
    travel_raw = row.get('Min of Travel', '') or row.get('MinOfTravel', '') or row.get('min_of_travel', '') or row.get('Travel Minutes', '') or row.get('Travel Time', '')
    if travel_raw is not None and str(travel_raw).strip() != '':
        m = re.search(r'(\d+)', str(travel_raw))
        travel_mins = int(m.group(1)) if m else None
    pickup_addr = str(row.get('Pickup Address', '') or row.get('Pickup_Address', '') or row.get('pickup_address', '') or row.get('Origin', '') or row.get('From Address', '') or row.get('From', '')).strip()
    dropoff_addr = str(row.get('Drop-off Address', '') or row.get('Dropoff Address', '') or row.get('dropoff_address', '') or row.get('Drop_off_Address', '') or row.get('Destination', '') or row.get('To Address', '')).strip()
    patient_key = ' '.join(patient.split())
    patient_id = f"{patient_key}|{dob}" if patient_key and dob else patient_key
    trips.append({
        'auth': auth,
        'date': date_str,
        'pickup': pickup,
        'pickupMins': to_mins(pickup),
        'zone': zone,
        'los': los,
        'isWR': bool(re.match(r'^(YES|Y|WR|WAIT|WAIT & RETURN|WAIT AND RETURN)$', wr, re.I)),
        'specialEquip': specEq,
        'patient': patient,
        'patientDob': dob,
        'patientId': patient_id,
        'travelMins': travel_mins,
        'pickupAddr': pickup_addr,
        'dropoffAddr': dropoff_addr,
    })

print('Parsed crew rows:', len(users))
print('Parsed vehicles:', len(vehicles))
print('Parsed trips:', len(trips))


Parsed crew rows: 64
Parsed vehicles: 17
Parsed trips: 64


In [7]:
def build_active_roster(date_str):
    dt = datetime.strptime(date_str, '%m/%d/%Y')
    dow = dt.weekday()  # Monday=0
    active = []
    next_id = 1
    for user in users:
        active_flag = user['status'] == 'Active' and bool(user['days'][dow])
        active.append({
            'id': next_id,
            'name': user['name'],
            'type': user['type'],
            'zone': user['zone'],
            'status': user['status'],
            'days': user['days'],
            'active': active_flag,
            'shiftStart': None,
            'shiftEnd': None,
            'lunchStart': None,
            'availFrom': 0,
            'assignments': [],
            'pairId': None,
            'vehicle': None,
            'hbPinned': False,
            'groupReservedUntil': None,
        })
        next_id += 1
    return active


def run_one_date(date_str):
    active_roster = build_active_roster(date_str)
    day_trips = [t for t in trips if t['date'] == date_str]
    day_trips = [t for t in day_trips if t['zone'] and t['pickup'] and t['los']]
    day_trips.sort(key=lambda x: x['pickupMins'])

    empState = {e['id']: e for e in active_roster}
    emtPairs = {}
    by_zone = defaultdict(list)
    for e in empState.values():
        if not e['active'] or e['type'] != 'EMT':
            continue
        by_zone[e['zone']].append(e)
    for zone, emts in by_zone.items():
        for i in range(0, len(emts) - 1, 2):
            a = emts[i]
            b = emts[i+1]
            pk = f"pair_{a['id']}_{b['id']}"
            a['pairId'] = b['pairId'] = pk
            emtPairs[pk] = {'key': pk, 'ids': [a['id'], b['id']], 'zone': zone, 'availFrom': 0, 'vehicle': None}
        if len(emts) % 2 == 1:
            emts[-1]['pairId'] = 'UNPAIRED'

    rnByZone = defaultdict(list)
    for e in empState.values():
        if e['active'] and e['type'] == 'RN':
            rnByZone[e['zone']].append(e)

    cctActive = {'SACRAMENTO': 0, 'SANTA CLARA': 0}
    results = {t['auth']: {**t, 'crew': [], 'crewName': None, 'vehicle': None, 'status': 'Unassigned', 'flags': [], 'candidateCrew': [], 'shiftStart': None, 'lunchStart': None} for t in day_trips}
    groupAssignments = {}
    patientGroups = defaultdict(list)
    for t in day_trips:
        if t.get('patientId'):
            patientGroups[t['patientId']].append(t)
    groupKeys = {pid for pid, lst in patientGroups.items() if len(lst) > 1}
    groupMasterAuths = set(
        sorted(lst, key=lambda x: x['pickupMins'])[0]['auth']
        for pid, lst in patientGroups.items() if len(lst) > 1
    )
    for t in day_trips:
        t['isWrMaster'] = t['isWR'] and t['auth'] in groupMasterAuths
    groupEndTimes = {pid: max(t['pickupMins'] for t in lst) for pid, lst in patientGroups.items() if len(lst) > 1}

    travel_cache = {}
    veh_shift_map = {}

    def get_group_assignment(trip):
        if not trip.get('patientId') or trip['patientId'] not in groupKeys:
            return None
        return groupAssignments.get(trip['patientId'])

    def save_group_assignment(trip, crew_ids, vehicle):
        if not trip.get('patientId') or trip['patientId'] not in groupKeys:
            return
        groupAssignments[trip['patientId']] = {'crewIds': crew_ids, 'vehicle': vehicle}

    def can_use_group_crew(trip, crew_ids):
        crew = [empState[id_] for id_ in crew_ids if id_ in empState]
        if len(crew) != len(crew_ids):
            results[trip['auth']]['flags'].append('WAIT/RETURN GROUP CREW MISSING MEMBER')
            return None
        blk = fallback_block(trip)
        invalid_members = []
        for e in crew:
            if e['zone'] != trip['zone'] or not e['active']:
                invalid_members.append(e)
                continue
            if not can_do_trip(e, trip, blk, ignore_group_reserved_until=True, ignore_lunch=True, ignore_avail_from=True):
                invalid_members.append(e)
        if invalid_members:
            reasons = []
            for e in invalid_members:
                if e['zone'] != trip['zone']:
                    reasons.append(f"{e['name'].split()[-1]} wrong zone")
                elif not e['active']:
                    reasons.append(f"{e['name'].split()[-1]} inactive")
                else:
                    reasons.append(f"{e['name'].split()[-1]} unavailable for W&R")
            results[trip['auth']]['flags'].append('WAIT/RETURN GROUP CREW UNAVAILABLE (' + ', '.join(reasons) + ')')
            return None
        return crew

    def apply_group_crew(trip, crew, vname, blk):
        reservation_end = get_wr_reservation_end(trip, blk, groupKeys, groupMasterAuths, groupEndTimes)
        for e in crew:
            if e['shiftStart'] is None:
                e['shiftStart'] = calc_shift_start(trip['pickupMins'], trip['travelMins'])
                e['shiftEnd'] = e['shiftStart'] + SHIFT_TOTAL
                e['lunchStart'] = calc_lunch_start(e['shiftStart'])
            e['assignments'].append({'auth': trip['auth'], 'los': trip['los'], 'pickup': trip['pickupMins'], 'blockEnd': reservation_end, 'isWR': trip['isWR'], 'vehicle': vname, 'dropoffAddr': trip['dropoffAddr'], 'pickupAddr': trip['pickupAddr'], 'blkMins': blk})
            e['availFrom'] = max(e['availFrom'], reservation_end)
            if groupEndTimes.get(trip.get('patientId')):
                e['groupReservedUntil'] = groupEndTimes[trip['patientId']]
        if vname and trip['isWR'] and groupEndTimes.get(trip.get('patientId')):
            claim_veh_for_shift(vname, trip['pickupMins'], reservation_end, crew[0]['id'], veh_shift_map)
        results[trip['auth']]['crew'] = [{'id': e['id'], 'name': e['name'], 'type': e['type']} for e in crew]
        results[trip['auth']]['crewName'] = ' / '.join(e['name'].split()[-1] for e in crew)
        results[trip['auth']]['vehicle'] = vname
        results[trip['auth']]['status'] = 'Assigned'
        if not vname:
            results[trip['auth']]['flags'].append('NO VEHICLE')

    def reuse_group_crew(trip, count, rule):
        group = get_group_assignment(trip)
        if not group:
            results[trip['auth']]['flags'].append('WAIT/RETURN NO SAVED CREW FOR PATIENT')
            return False
        crew = can_use_group_crew(trip, group['crewIds'])
        if not crew or len(crew) < count:
            if len(group['crewIds']) < count:
                results[trip['auth']]['flags'].append('WAIT/RETURN SAVED CREW INCOMPLETE')
            return False
        vname = group['vehicle']
        if not vname:
            results[trip['auth']]['flags'].append('WAIT/RETURN SAVED CREW HAS NO VEHICLE')
            return False
        chosen = crew[:count]
        apply_group_crew(trip, chosen, vname, fallback_block(trip))
        return True

    def set_shift(e, pm, travel):
        if e['shiftStart'] is not None or e['hbPinned']:
            return
        e['shiftStart'] = calc_shift_start(pm, travel)
        e['shiftEnd'] = e['shiftStart'] + SHIFT_TOTAL
        e['lunchStart'] = calc_lunch_start(e['shiftStart'])

    def eff_crew(trip, rule):
        return max(2, rule['crew']) if trip.get('specialEquip') else rule['crew']

    def doNE(trip, count, rule):
        pm = trip['pickupMins']
        blk = get_wr_block(trip, groupKeys, groupMasterAuths, groupEndTimes)
        if reuse_group_crew(trip, count, rule):
            return True
        all_elig = [e for e in empState.values() if e['active'] and e['type'] == 'Non-EMT' and e['zone'] == trip['zone'] and can_do_trip(e, trip, blk)]
        committed = sorted([e for e in all_elig if e['shiftStart'] is not None], key=lambda x: x['availFrom'])
        fresh = [e for e in all_elig if e['shiftStart'] is None]
        pool = committed + fresh
        if len(pool) < count:
            return False
        chosen = pool[:count]
        for e in chosen:
            if e['shiftStart'] is None:
                travel = None if e['assignments'] else get_base_to_pickup_mins(trip, travel_cache)
                set_shift(e, pm, travel)
        vname = None
        if count == 1:
            e = chosen[0]
            e['vehicle'] = get_crew_vehicle(e, trip, blk, rule, vehicles, veh_shift_map)
            if not e['vehicle']:
                results[trip['auth']]['candidateCrew'] = [{'id': e['id'], 'name': e['name'], 'type': 'Non-EMT'} for e in chosen]
                results[trip['auth']]['flags'].append('CREW AVAILABLE BUT NO VEHICLE (' + ' or '.join(rule['vehicles']) + (" (bariatric)" if rule['bariatric'] else '') + ')')
                return False
            vname = e['vehicle']
        else:
            lead = chosen[0]
            lead['vehicle'] = get_crew_vehicle(lead, trip, blk, rule, vehicles, veh_shift_map)
            for e in chosen:
                e['vehicle'] = lead['vehicle']
            if not lead['vehicle']:
                results[trip['auth']]['candidateCrew'] = [{'id': e['id'], 'name': e['name'], 'type': 'Non-EMT'} for e in chosen]
                results[trip['auth']]['flags'].append('CREW AVAILABLE BUT NO VEHICLE (' + ' or '.join(rule['vehicles']) + (" (bariatric)" if rule['bariatric'] else '') + ')')
                return False
            vname = lead['vehicle']
        for e in chosen:
            if e['shiftStart'] is None:
                set_shift(e, pm, trip['travelMins'])
            e['assignments'].append({'auth': trip['auth'], 'los': trip['los'], 'pickup': pm, 'blockEnd': pm + blk, 'isWR': trip['isWR'], 'vehicle': vname, 'dropoffAddr': trip['dropoffAddr'], 'pickupAddr': trip['pickupAddr'], 'blkMins': blk})
            e['availFrom'] = pm + blk
        results[trip['auth']]['crew'] = [{'id': e['id'], 'name': e['name'], 'type': 'Non-EMT'} for e in chosen]
        results[trip['auth']]['crewName'] = ' / '.join(e['name'].split()[-1] for e in chosen)
        results[trip['auth']]['vehicle'] = vname
        results[trip['auth']]['status'] = 'Assigned'
        if not vname:
            results[trip['auth']]['flags'].append('NO VEHICLE')
        if trip.get('patientId'):
            save_group_assignment(trip, [e['id'] for e in chosen], vname)
        return True

    def doEMT(trip, rule):
        pm = trip['pickupMins']
        blk = get_wr_block(trip, groupKeys, groupMasterAuths, groupEndTimes)
        if reuse_group_crew(trip, 2, rule):
            return True
        elig_pairs = [p for p in emtPairs.values() if p['zone'] == trip['zone'] and can_do_trip(empState[p['ids'][0]], trip, blk) and can_do_trip(empState[p['ids'][1]], trip, blk) and pm >= p['availFrom']]
        pair = next((p for p in elig_pairs if empState[p['ids'][0]]['shiftStart'] is not None), None)
        if not pair:
            pair = elig_pairs[0] if elig_pairs else None
        if not pair:
            return False
        a = empState[pair['ids'][0]]
        b = empState[pair['ids'][1]]
        travel = None if a['assignments'] else get_base_to_pickup_mins(trip, travel_cache)
        set_shift(a, pm, travel)
        set_shift(b, pm, travel)
        pair['availFrom'] = pm + blk
        if not pair.get('vehicle') or not is_veh_free_for_shift(pair['vehicle'], pm, pm + blk, a['id'], veh_shift_map):
            pair['vehicle'] = get_crew_vehicle(a, trip, blk, rule, vehicles, veh_shift_map)
            a['vehicle'] = b['vehicle'] = pair['vehicle']
        if not pair['vehicle']:
            results[trip['auth']]['candidateCrew'] = [{'id': e['id'], 'name': e['name'], 'type': 'EMT'} for e in (a,b)]
            results[trip['auth']]['flags'].append('CREW AVAILABLE BUT NO VEHICLE (' + ' or '.join(rule['vehicles']) + ')')
            return False
        vname = pair['vehicle']
        for e in (a, b):
            e['assignments'].append({'auth': trip['auth'], 'los': trip['los'], 'pickup': pm, 'blockEnd': pm + blk, 'isWR': trip['isWR'], 'vehicle': vname, 'dropoffAddr': trip['dropoffAddr'], 'pickupAddr': trip['pickupAddr'], 'blkMins': blk})
            e['availFrom'] = pm + blk
        results[trip['auth']]['crew'] = [{'id': a['id'], 'name': a['name'], 'type': 'EMT'}, {'id': b['id'], 'name': b['name'], 'type': 'EMT'}]
        results[trip['auth']]['crewName'] = f"{a['name'].split()[-1]} / {b['name'].split()[-1]}"
        results[trip['auth']]['vehicle'] = vname
        results[trip['auth']]['status'] = 'Assigned'
        if not vname:
            results[trip['auth']]['flags'].append('NO VEHICLE')
        if trip.get('patientId'):
            save_group_assignment(trip, [a['id'], b['id']], vname)
        return True

    def doCCT(trip, rule):
        pm = trip['pickupMins']
        blk = get_wr_block(trip, groupKeys, groupMasterAuths, groupEndTimes)
        if reuse_group_crew(trip, 3, rule):
            return True
        if cctActive[trip['zone']] >= 1:
            results[trip['auth']]['flags'].append('CCT CAP WARN')
        pair = None
        for p in emtPairs.values():
            if p['zone'] != trip['zone']:
                continue
            a = empState[p['ids'][0]]
            b = empState[p['ids'][1]]
            if a and b and can_do_trip(a, trip, blk) and can_do_trip(b, trip, blk) and pm >= p['availFrom']:
                pair = p
                break
        if not pair:
            return False
        rns = [e for e in rnByZone[trip['zone']] if can_do_trip(e, trip, blk)]
        rns.sort(key=lambda x: x['availFrom'])
        if not rns:
            return False
        a = empState[pair['ids'][0]]
        b = empState[pair['ids'][1]]
        rn = rns[0]
        pair['availFrom'] = pm + blk
        cctActive[trip['zone']] += 1
        travel = None if a['assignments'] else get_base_to_pickup_mins(trip, travel_cache)
        set_shift(a, pm, travel)
        set_shift(b, pm, travel)
        set_shift(rn, pm, travel)
        if not pair.get('vehicle') or not is_veh_free_for_shift(pair['vehicle'], pm, pm + blk, a['id'], veh_shift_map):
            pair['vehicle'] = get_crew_vehicle(a, trip, blk, {'bariatric': False, 'vehicles': rule['vehicles']}, vehicles, veh_shift_map)
            a['vehicle'] = b['vehicle'] = rn['vehicle'] = pair['vehicle']
        if not pair['vehicle']:
            results[trip['auth']]['candidateCrew'] = [{'id': e['id'], 'name': e['name'], 'type': e['type']} for e in (a, b, rn)]
            results[trip['auth']]['flags'].append('CREW AVAILABLE BUT NO VEHICLE (' + ' or '.join(rule['vehicles']) + ')')
            return False
        vname = pair['vehicle']
        for e in (a, b, rn):
            e['assignments'].append({'auth': trip['auth'], 'los': trip['los'], 'pickup': pm, 'blockEnd': pm + blk, 'isWR': trip['isWR'], 'vehicle': vname, 'dropoffAddr': trip['dropoffAddr'], 'pickupAddr': trip['pickupAddr'], 'blkMins': blk})
            e['availFrom'] = pm + blk
        results[trip['auth']]['crew'] = [{'id': a['id'], 'name': a['name'], 'type': 'EMT'}, {'id': b['id'], 'name': b['name'], 'type': 'EMT'}, {'id': rn['id'], 'name': rn['name'], 'type': rn['type']}]
        results[trip['auth']]['crewName'] = f"{a['name'].split()[-1]} / {b['name'].split()[-1]} / {rn['name'].split()[-1]}"
        results[trip['auth']]['vehicle'] = vname
        results[trip['auth']]['status'] = 'Assigned'
        if not vname:
            results[trip['auth']]['flags'].append('NO VEHICLE')
        if trip.get('patientId'):
            save_group_assignment(trip, [a['id'], b['id'], rn['id']], vname)
        return True

    def assign_trip(trip):
        if trip.get('patientId') and trip['patientId'] in groupKeys and trip['auth'] not in groupMasterAuths and trip['patientId'] not in groupAssignments:
            return
        rule = LOS_RULES.get(trip['los'])
        if not rule:
            return
        if not check_location_and_capability(trip['zone'], trip['los']):
            results[trip['auth']]['status'] = 'Blocked'
            results[trip['auth']]['flags'].append(f"{trip['los']} not authorised for {trip['zone']} base")
            return
        if trip['los'] == 'CCT':
            blkCCT = fallback_block(trip)
            has_pair = any(p['zone'] == trip['zone'] and can_do_trip(empState[p['ids'][0]], trip, blkCCT) and can_do_trip(empState[p['ids'][1]], trip, blkCCT) and trip['pickupMins'] >= p['availFrom'] for p in emtPairs.values())
            has_rn = any(can_do_trip(e, trip, blkCCT) for e in rnByZone[trip['zone']])
            if not doCCT(trip, rule):
                if not has_pair:
                    results[trip['auth']]['flags'].append('No EMT pair available at ' + f"{trip['pickup']}" )
                if not has_rn:
                    results[trip['auth']]['flags'].append('No RN available at ' + f"{trip['pickup']}")
            return
        if use_emt_for_trip(trip, rule):
            if not doEMT(trip, rule):
                pm = trip['pickupMins']
                pairs = [p for p in emtPairs.values() if p['zone'] == trip['zone']]
                if not pairs:
                    results[trip['auth']]['flags'].append('No EMT pairs in ' + trip['zone'])
                else:
                    for p in pairs:
                        a = empState[p['ids'][0]]
                        b = empState[p['ids'][1]]
                        if pm < p['availFrom']:
                            results[trip['auth']]['flags'].append((p.get('crewName') or p['key']) + ' busy until ' + f"{p['availFrom'] // 60:02d}:{p['availFrom'] % 60:02d}")
                        elif not (can_do_trip(a, trip, fallback_block(trip)) and can_do_trip(b, trip, fallback_block(trip))):
                            results[trip['auth']]['flags'].append((p.get('crewName') or p['key']) + ' outside shift')
                        else:
                            results[trip['auth']]['flags'].append('All EMT pairs busy at ' + f"{trip['pickup']}")
            return
        needed = eff_crew(trip, rule)
        if not doNE(trip, needed, rule):
            pm = trip['pickupMins']
            blk = get_wr_block(trip, groupKeys, groupMasterAuths, groupEndTimes)
            ne_pool = [e for e in empState.values() if e['active'] and e['type'] == 'Non-EMT' and e['zone'] == trip['zone']]
            avail = [e for e in ne_pool if can_do_trip(e, trip, blk)]
            started = [e for e in ne_pool if e['shiftStart'] is not None]
            all_shifts_ended = bool(started) and all(pm >= e['shiftEnd'] for e in started)
            latest_shift_end = max(e['shiftEnd'] for e in started) if started else None
            attempted_fallback = can_fallback_to_emt(trip) and doEMT(trip, rule)
            if attempted_fallback:
                results[trip['auth']]['flags'].append('EMT used to cover transport')
                return
            if trip['los'] in NO_EMT_FALLBACK:
                results[trip['auth']]['flags'].append('No EMT fallback for wheelchair transport')
            if not ne_pool:
                results[trip['auth']]['flags'].append('No Non-EMT staff in ' + trip['zone'])
                results[trip['auth']]['shiftGap'] = True
            elif all_shifts_ended:
                results[trip['auth']]['flags'].append('All shifts ended by ' + f"{latest_shift_end // 60:02d}:{latest_shift_end % 60:02d}" + ' — PM crew needed for ' + trip['pickup'] + ' pickup')
                results[trip['auth']]['shiftGap'] = True
            elif not avail:
                busy_workers = sorted([e for e in started if pm < e['shiftEnd']], key=lambda x: x['availFrom'])
                next_free = min((e['availFrom'] for e in busy_workers), default=None)
                results[trip['auth']]['flags'].append('All ' + str(len(ne_pool)) + ' Non-EMT busy' + ('' if next_free is None else ' — next free ' + f"{next_free // 60:02d}:{next_free % 60:02d}"))
            else:
                results[trip['auth']]['flags'].append('Need ' + str(needed) + ' Non-EMT, only ' + str(len(avail)) + ' available at ' + trip['pickup'])
            emt_pair_list = [p for p in emtPairs.values() if p['zone'] == trip['zone']]
            if not emt_pair_list:
                results[trip['auth']]['flags'].append('No EMT pairs for fallback either')
            else:
                emt_all_ended = all((not empState[p['ids'][0]]['shiftEnd'] or pm >= empState[p['ids'][0]]['shiftEnd']) and (not empState[p['ids'][1]]['shiftEnd'] or pm >= empState[p['ids'][1]]['shiftEnd']) for p in emt_pair_list)
                if emt_all_ended:
                    results[trip['auth']]['flags'].append('All EMT shifts also ended')
                else:
                    results[trip['auth']]['flags'].append('All EMT pairs busy at ' + trip['pickup'])

    for trip in day_trips:
        assign_trip(trip)

    # Assign remaining W&R group legs if master was assigned
    for pid, group in patientGroups.items():
        if len(group) <= 1:
            continue
        group_sorted = sorted(group, key=lambda x: x['pickupMins'])
        for trip in group_sorted:
            if results[trip['auth']]['status'] != 'Unassigned':
                continue
            if not get_group_assignment(trip):
                continue
            rule = LOS_RULES.get(trip['los'])
            if not rule:
                continue
            if trip['los'] == 'CCT':
                doCCT(trip, rule)
            elif rule['emt'] > 0:
                doEMT(trip, rule)
            else:
                doNE(trip, eff_crew(trip, rule), rule)

    # Build summary result scope
    for p in emtPairs.values():
        a = empState[p['ids'][0]]
        b = empState[p['ids'][1]]
        if a and b:
            default = a['name'].split()[-1] + ' / ' + b['name'].split()[-1]
            p['crewName'] = default
            a['crewName'] = b['crewName'] = p['crewName']

    for e in empState.values():
        if e['active'] and e['type'] != 'EMT':
            e['crewName'] = e.get('name').split()[-1]
        if e['active'] and e['type'] == 'RN':
            e['crewName'] = e.get('name').split()[-1]

    for r in results.values():
        if not r['crewName'] and r.get('crew'):
            first = empState.get(r['crew'][0]['id'])
            r['crewName'] = first['crewName'] if first else ' / '.join(c['name'].split()[-1] for c in r['crew'])
        req = LOS_RULES.get(r['los'], {}).get('crew', 1)
        if r['status'] == 'Assigned' and len(r.get('crew', [])) < req:
            r['status'] = 'Partial'
            r['flags'].append('UNDERSTAFFED')
        if r['status'] == 'Assigned':
            r['flags'] = [f for f in r['flags'] if f != 'WAIT/RETURN NO SAVED CREW FOR PATIENT']
        if r.get('crew'):
            e = empState.get(r['crew'][0]['id'])
            if e:
                r['shiftStart'] = e['shiftStart']
                r['lunchStart'] = e['lunchStart']

    return {
        'results': list(results.values()),
        'empState': empState,
        'emtPairs': emtPairs,
        'rnByZone': rnByZone,
    }

mapping = run_one_date('04/17/2026')
assigned = [r for r in mapping['results'] if r['status'] == 'Assigned']
unassigned = [r for r in mapping['results'] if r['status'] == 'Unassigned']
partial = [r for r in mapping['results'] if r['status'] == 'Partial']
blocked = [r for r in mapping['results'] if r['status'] == 'Blocked']
print('Total trips on 04/17/2026:', len(mapping['results']))
print('Assigned:', len(assigned))
print('Partial:', len(partial))
print('Unassigned:', len(unassigned))
print('Blocked:', len(blocked))
print('Sample assigned trips:', [(r['auth'], r['crewName'], r['vehicle'], r['flags']) for r in assigned[:10]])
print('Sample unassigned trips:', [(r['auth'], r['flags']) for r in unassigned[:10]])


Total trips on 04/17/2026: 64
Assigned: 10
Partial: 0
Unassigned: 54
Blocked: 0
Sample assigned trips: [('MHQLD2ML', 'Izmarai / Seidell', 'M20', []), ('67O5Z2J2', 'Izmarai / Seidell', 'M20', []), ('K2LZDFQR', 'Tan / Ceja', 'M23', []), ('YCFUFVJP', 'Izmarai / Seidell', 'M20', []), ('L36YJLDR', 'Tan / Ceja', 'M23', []), ('3Q6D3KEB', 'Goforth / Abohasan', 'M20', []), ('TWGQXVOY', 'Izmarai / Seidell', 'M20', []), ('5MVMNFQ2', 'Tan / Ceja', 'M23', []), ('3VNDTKGZ', 'Goforth / Abohasan', 'M20', []), ('HGVGBLAF', 'Gill / Narayan', 'M20', ['WAIT/RETURN GROUP CREW UNAVAILABLE (Tan unavailable for W&R, Ceja unavailable for W&R)'])]
Sample unassigned trips: [('YBNUNF6W', ['WAIT/RETURN NO SAVED CREW FOR PATIENT', 'No EMT fallback for wheelchair transport', 'No Non-EMT staff in SACRAMENTO', 'All EMT shifts also ended']), ('JEAOJFM6', ['WAIT/RETURN NO SAVED CREW FOR PATIENT', 'No EMT fallback for wheelchair transport', 'No Non-EMT staff in SACRAMENTO', 'All EMT shifts also ended']), ('2Z7XDFPB', ['W

In [2]:
%pip install pandas

  Using cached pandas-3.0.2-cp314-cp314-win_amd64.whl.metadata (19 kB)
  Using cached numpy-2.4.4-cp314-cp314-win_amd64.whl.metadata (6.6 kB)
  Using cached tzdata-2026.1-py2.py3-none-any.whl.metadata (1.4 kB)
Using cached pandas-3.0.2-cp314-cp314-win_amd64.whl (9.9 MB)
Using cached numpy-2.4.4-cp314-cp314-win_amd64.whl (12.4 MB)
Using cached tzdata-2026.1-py2.py3-none-any.whl (348 kB)

   ---------------------------------------- 0/3 [tzdata]
   ---------------------------------------- 0/3 [tzdata]
   ---------------------------------------- 0/3 [tzdata]
   ---------------------------------------- 0/3 [tzdata]
   ---------------------------------------- 0/3 [tzdata]
   ---------------------------------------- 0/3 [tzdata]
   ---------------------------------------- 0/3 [tzdata]
   ---------------------------------------- 0/3 [tzdata]
   ---------------------------------------- 0/3 [tzdata]
   ---------------------------------------- 0/3 [tzdata]
   ------------- -----------------------


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
import subprocess
import json

for cmd in [['node','--version'], ['node','-e','console.log("NODE_OK")'], ['node','-e','try{require("jsdom");console.log("JSDOM_OK");}catch(e){console.log("JSDOM_MISSING",e.message);}']]:
    try:
        completed = subprocess.run(cmd, capture_output=True, text=True, timeout=10)
        print('CMD:', ' '.join(cmd))
        print('RETURN', completed.returncode)
        print('STDOUT:', completed.stdout.strip())
        print('STDERR:', completed.stderr.strip())
    except Exception as e:
        print('CMD ERROR:', ' '.join(cmd), repr(e))


CMD: node --version
RETURN 0
STDOUT: v25.2.1
STDERR: 
CMD: node -e console.log("NODE_OK")
RETURN 0
STDOUT: NODE_OK
STDERR: 
CMD: node -e try{require("jsdom");console.log("JSDOM_OK");}catch(e){console.log("JSDOM_MISSING",e.message);}
RETURN 0
STDOUT: JSDOM_MISSING Cannot find module 'jsdom'
Require stack:
- c:\App\Mapping_oLD\[eval]
STDERR: 


In [9]:
for cmd in [['npm','--version'], ['npm','install','jsdom']]:
    try:
        completed = subprocess.run(cmd, capture_output=True, text=True, timeout=120)
        print('CMD:', ' '.join(cmd))
        print('RETURN', completed.returncode)
        print('STDOUT:', completed.stdout.strip())
        print('STDERR:', completed.stderr.strip())
    except Exception as e:
        print('CMD ERROR:', ' '.join(cmd), repr(e))


CMD ERROR: npm --version FileNotFoundError(2, 'The system cannot find the file specified', None, 2, None)
CMD ERROR: npm install jsdom FileNotFoundError(2, 'The system cannot find the file specified', None, 2, None)


In [11]:
for cmd in [['corepack','--version'], ['yarn','--version'], ['pnpm','--version']]:
    try:
        completed = subprocess.run(cmd, capture_output=True, text=True, timeout=30)
        print('CMD:', ' '.join(cmd))
        print('RETURN', completed.returncode)
        print('STDOUT:', completed.stdout.strip())
        print('STDERR:', completed.stderr.strip())
    except Exception as e:
        print('CMD ERROR:', ' '.join(cmd), repr(e))


CMD ERROR: corepack --version FileNotFoundError(2, 'The system cannot find the file specified', None, 2, None)
CMD ERROR: yarn --version FileNotFoundError(2, 'The system cannot find the file specified', None, 2, None)
CMD ERROR: pnpm --version FileNotFoundError(2, 'The system cannot find the file specified', None, 2, None)


In [14]:
for pkg in ['quickjs', 'py_mini_racer', 'js2py']:
    try:
        completed = subprocess.run([sys.executable, '-m', 'pip', 'install', pkg], capture_output=True, text=True, timeout=120)
        print('INSTALL', pkg, 'RETURN', completed.returncode)
        print('STDOUT', completed.stdout.strip()[:1000])
        print('STDERR', completed.stderr.strip()[:1000])
    except Exception as e:
        print('INSTALL ERROR', pkg, repr(e))


INSTALL quickjs RETURN 1
STDOUT Collecting quickjs
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
Failed to build quickjs
STDERR error: subprocess-exited-with-error
  
  Ã— Building wheel for quickjs (pyproject.toml) did not run successfully.
  â”‚ exit code: 1
  â•°â”€> [12 lines of output]
      running bdist_wheel
      running build
      running build_py
      creating build\lib.win-amd64-cpython-314\quickjs
      copying quickjs\__init__.py -> build\lib.win-amd64-cpython-314\quickjs
      running build_ext
      building '_quickjs' extension
      creating build\temp.win-amd64-cpython-314\Release
      creating build\temp.win-amd64-cpython-314\Release\upstream-quickjs
      "C:\Program Files (x86)\Micr

In [13]:
import sys


In [15]:
from pathlib import Path
import re
text = Path('Mapping.html').read_text(encoding='utf-8')
for pat in [r'document\.', r'window\.', r'fetch\(', r'localStorage', r'navigator', r'XMLHttpRequest', r'alert\(']:
    print(pat, len(re.findall(pat, text)))


document\. 76
window\. 3
fetch\( 3
localStorage 16
navigator 0
XMLHttpRequest 0
alert\( 0


In [16]:
from py_mini_racer import py_mini_racer
import json
from pathlib import Path
import csv

# Load the Mapping.html JS section
text = Path('Mapping.html').read_text(encoding='utf-8')
script_start = text.index('<script>\n', text.index('xlsx.full.min.js')) + len('<script>\n')
script_end = text.index('</script>', script_start)
js_script = text[script_start:script_end]

# Build global browser stubs and helper functions
js_prelude = r'''
var window = this;
var navigator = {};
var location = {protocol:'file:'};
var localStorage = {data:{}, getItem:function(k){return this.data[k]||null;}, setItem:function(k,v){this.data[k]=String(v);}, removeItem:function(k){delete this.data[k];}};
var document = { getElementById:function(id){return {value:'',checked:false,style:{display:''},textContent:'', innerHTML:'', addEventListener:function(){}, removeEventListener:function(){}, appendChild:function(){}, querySelector:function(){return null;}, querySelectorAll:function(){return [];}, setAttribute:function(){}, getAttribute:function(){return null;}, focus:function(){}, selectedIndex:0, options:[], text:''};}, querySelector:function(){return null;}, querySelectorAll:function(){return [];}, createElement:function(){return {style:{}, value:'', checked:false, appendChild:function(){}, addEventListener:function(){}, removeEventListener:function(){}, setAttribute:function(){}, getAttribute:function(){return null}}; }};
var FileReader = function(){ this.readAsArrayBuffer = function(){}; };
var XLSX = {read:function(){return {SheetNames:[], Sheets:{}};}, utils:{sheet_to_json:function(){return [];}}};
function toast(msg,type){}
function renderRoster(){}
function renderVehicles(){}
function loadSettings(){}
function updateDateUI(){}
function renderCurrentView(){}
function updateNavBadges(){}
function updateHbStatus(){}
function updateCacheCount(){}
function switchTab(){}
function preFetchDistances(){return Promise.resolve();}
function setTimeout(fn, time){fn();}
function alert(msg){}
function fetch(){return Promise.resolve({ok:true,json:()=>Promise.resolve({}),text:()=>Promise.resolve('')});}
'''

ctx = py_mini_racer.MiniRacer()
ctx.eval(js_prelude)
ctx.eval(js_script)

# Load CSVs as row lists
with open('Users.csv', newline='', encoding='utf-8-sig') as f:
    users_rows = list(csv.DictReader(f))
with open('Vehicles.csv', newline='', encoding='utf-8-sig') as f:
    vehicles_rows = list(csv.DictReader(f))
with open('Trips.csv', newline='', encoding='utf-8-sig') as f:
    trips_rows = list(csv.DictReader(f))

ctx.eval('var usersRaw = %s;' % json.dumps(users_rows))
ctx.eval('var vehiclesRaw = %s;' % json.dumps(vehicles_rows))
ctx.eval('var tripsRaw = %s;' % json.dumps(trips_rows))

ctx.eval('parseUsers(usersRaw); parseVehicles(vehiclesRaw); parseTrips(tripsRaw);')

result = ctx.eval('var res = mappings["04/17/2026"] || mappings[allDates[0]]; var assigned=res.results.filter(r=>r.status===\"Assigned\").length; var unassigned=res.results.filter(r=>r.status===\"Unassigned\").length; var partial=res.results.filter(r=>r.status===\"Partial\").length; var blocked=res.results.filter(r=>r.status===\"Blocked\").length; ({assigned, unassigned, partial, blocked, total: res.results.length, sampleAssigned: res.results.filter(r=>r.status===\"Assigned\").slice(0,10).map(r=>({auth:r.auth, crewName:r.crewName, vehicle:r.vehicle, flags:r.flags})), sampleUnassigned: res.results.filter(r=>r.status===\"Unassigned\").slice(0,10).map(r=>({auth:r.auth, flags:r.flags}))});')
print(result)


JSEvalException: Uncaught TypeError: Cannot read property 'protocol' of undefined at undefined:764:66
TypeError: Cannot read property 'protocol' of undefined
    at <anonymous>:764:67

In [17]:
from py_mini_racer import py_mini_racer
import json
from pathlib import Path
import csv

# Load the Mapping.html JS section
text = Path('Mapping.html').read_text(encoding='utf-8')
script_start = text.index('<script>\n', text.index('xlsx.full.min.js')) + len('<script>\n')
script_end = text.index('</script>', script_start)
js_script = text[script_start:script_end]

# Build global browser stubs and helper functions
js_prelude = r'''
var window = this;
var navigator = {};
var location = {protocol:'file:'};
var localStorage = {data:{}, getItem:function(k){return this.data[k]||null;}, setItem:function(k,v){this.data[k]=String(v);}, removeItem:function(k){delete this.data[k];}};
var document = { getElementById:function(id){return {value:'',checked:false,style:{display:''},textContent:'', innerHTML:'', addEventListener:function(){}, removeEventListener:function(){}, appendChild:function(){}, querySelector:function(){return null;}, querySelectorAll:function(){return [];}, setAttribute:function(){}, getAttribute:function(){return null;}, focus:function(){}, selectedIndex:0, options:[], text:''};}, querySelector:function(){return null;}, querySelectorAll:function(){return [];}, createElement:function(){return {style:{}, value:'', checked:false, appendChild:function(){}, addEventListener:function(){}, removeEventListener:function(){}, setAttribute:function(){}, getAttribute:function(){return null}}; }};
var FileReader = function(){ this.readAsArrayBuffer = function(){}; };
var XLSX = {read:function(){return {SheetNames:[], Sheets:{}};}, utils:{sheet_to_json:function(){return [];}}};
function toast(msg,type){}
function renderRoster(){}
function renderVehicles(){}
function loadSettings(){}
function updateDateUI(){}
function renderCurrentView(){}
function updateNavBadges(){}
function updateHbStatus(){}
function updateCacheCount(){}
function switchTab(){}
function preFetchDistances(){return Promise.resolve();}
function setTimeout(fn, time){fn();}
function alert(msg){}
function fetch(){return Promise.resolve({ok:true,json:()=>Promise.resolve({}),text:()=>Promise.resolve('')});}
'''

ctx = py_mini_racer.MiniRacer()
ctx.eval(js_prelude)
ctx.eval(js_script)

with open('Users.csv', newline='', encoding='utf-8-sig') as f:
    users_rows = list(csv.DictReader(f))
with open('Vehicles.csv', newline='', encoding='utf-8-sig') as f:
    vehicles_rows = list(csv.DictReader(f))
with open('Trips.csv', newline='', encoding='utf-8-sig') as f:
    trips_rows = list(csv.DictReader(f))

ctx.eval('var usersRaw = %s;' % json.dumps(users_rows))
ctx.eval('var vehiclesRaw = %s;' % json.dumps(vehicles_rows))
ctx.eval('var tripsRaw = %s;' % json.dumps(trips_rows))
ctx.eval('parseUsers(usersRaw); parseVehicles(vehiclesRaw); parseTrips(tripsRaw);')

result = ctx.eval('var res = mappings["04/17/2026"] || mappings[allDates[0]]; var assigned=res.results.filter(r=>r.status===\"Assigned\").length; var unassigned=res.results.filter(r=>r.status===\"Unassigned\").length; var partial=res.results.filter(r=>r.status===\"Partial\").length; var blocked=res.results.filter(r=>r.status===\"Blocked\").length; ({assigned, unassigned, partial, blocked, total: res.results.length, sampleAssigned: res.results.filter(r=>r.status===\"Assigned\").slice(0,10).map(r=>({auth:r.auth, crewName:r.crewName, vehicle:r.vehicle, flags:r.flags})), sampleUnassigned: res.results.filter(r=>r.status===\"Unassigned\").slice(0,10).map(r=>({auth:r.auth, flags:r.flags}))});')
print(result)


JSEvalException: Uncaught ReferenceError: clearTimeout is not defined at undefined:2195:118
ReferenceError: clearTimeout is not defined
    at toast (<anonymous>:2195:119)
    at <anonymous>:2224:16
    at setTimeout (<anonymous>:20:31)
    at <anonymous>:2224:1

In [33]:
from py_mini_racer import py_mini_racer
import json
from pathlib import Path
import csv

text = Path('Mapping.html').read_text(encoding='utf-8')
script_start = text.index('<script>\n', text.index('xlsx.full.min.js')) + len('<script>\n')
script_end = text.index('</script>', script_start)
js_script = text[script_start:script_end]

js_prelude = r'''
var window = this;
var navigator = {};
var location = {protocol:'file:'};
var localStorage = {data:{}, getItem:function(k){return this.data[k]||null;}, setItem:function(k,v){this.data[k]=String(v);}, removeItem:function(k){delete this.data[k];}};
var document = { getElementById:function(id){return {value:'',checked:false,style:{display:''},textContent:'', innerHTML:'', addEventListener:function(){}, removeEventListener:function(){}, appendChild:function(){}, querySelector:function(){return null;}, querySelectorAll:function(){return [];}, setAttribute:function(){}, getAttribute:function(){return null;}, focus:function(){}, selectedIndex:0, options:[], text:''};}, querySelector:function(){return null;}, querySelectorAll:function(){return [];}, createElement:function(){return {style:{}, value:'', checked:false, appendChild:function(){}, addEventListener:function(){}, removeEventListener:function(){}, setAttribute:function(){}, getAttribute:function(){return null}}; }};
var FileReader = function(){ this.readAsArrayBuffer = function(){}; };
var XLSX = {read:function(){return {SheetNames:[], Sheets:{}};}, utils:{sheet_to_json:function(){return [];}}};
function toast(msg,type){}
function renderRoster(){}
function renderVehicles(){}
function loadSettings(){}
function updateDateUI(){}
function renderCurrentView(){}
function updateNavBadges(){}
function updateHbStatus(){}
function updateCacheCount(){}
function switchTab(){}
function preFetchDistances(){return Promise.resolve();}
function setTimeout(fn, time){fn();return 0;}
function clearTimeout(id){}
function alert(msg){}
function fetch(){return Promise.resolve({ok:true,json:()=>Promise.resolve({}),text:()=>Promise.resolve('')});}
'''

ctx = py_mini_racer.MiniRacer()
ctx.eval(js_prelude)
ctx.eval(js_script)

with open('Users.csv', newline='', encoding='utf-8-sig') as f:
    users_rows = list(csv.DictReader(f))
with open('Vehicles.csv', newline='', encoding='utf-8-sig') as f:
    vehicles_rows = list(csv.DictReader(f))
with open('Trips.csv', newline='', encoding='utf-8-sig') as f:
    trips_rows = list(csv.DictReader(f))

ctx.eval('var usersRaw = %s;' % json.dumps(users_rows))
ctx.eval('var vehiclesRaw = %s;' % json.dumps(vehicles_rows))
ctx.eval('var tripsRaw = %s;' % json.dumps(trips_rows))
ctx.eval('parseUsers(usersRaw); parseVehicles(vehiclesRaw); parseTrips(tripsRaw);')

result = ctx.eval('var res = mappings["04/17/2026"] || mappings[allDates[0]]; var assigned=res.results.filter(r=>r.status===\"Assigned\").length; var unassigned=res.results.filter(r=>r.status===\"Unassigned\").length; var partial=res.results.filter(r=>r.status===\"Partial\").length; var blocked=res.results.filter(r=>r.status===\"Blocked\").length; ({assigned, unassigned, partial, blocked, total: res.results.length, sampleAssigned: res.results.filter(r=>r.status===\"Assigned\").slice(0,10).map(r=>({auth:r.auth, crewName:r.crewName, vehicle:r.vehicle, flags:r.flags})), sampleUnassigned: res.results.filter(r=>r.status===\"Unassigned\").slice(0,10).map(r=>({auth:r.auth, flags:r.flags}))});')
print(result)


In [34]:
result_json = ctx.eval("JSON.stringify((function(){var res = mappings['04/17/2026'] || mappings[allDates[0]]; return {assigned: res.results.filter(r=>r.status==='Assigned').length, unassigned: res.results.filter(r=>r.status==='Unassigned').length, partial: res.results.filter(r=>r.status==='Partial').length, blocked: res.results.filter(r=>r.status==='Blocked').length, total: res.results.length, sampleAssigned: res.results.filter(r=>r.status==='Assigned').slice(0,10).map(r=>({auth:r.auth, crewName:r.crewName, vehicle:r.vehicle, flags:r.flags})), sampleUnassigned: res.results.filter(r=>r.status==='Unassigned').slice(0,10).map(r=>({auth:r.auth, flags:r.flags}))};})())")
print(result_json)


{"assigned":43,"unassigned":21,"partial":0,"blocked":0,"total":64,"sampleAssigned":[{"auth":"YBNUNF6W","crewName":"Seidell","vehicle":"WC005","flags":[]},{"auth":"JEAOJFM6","crewName":"Seidell","vehicle":"WC005","flags":[]},{"auth":"2Z7XDFPB","crewName":"Ceja","vehicle":"GW002","flags":[]},{"auth":"RHIJPF7D","crewName":"Seidell","vehicle":"WC005","flags":[]},{"auth":"JSWIJ3BV","crewName":"Goforth","vehicle":"GW003","flags":[]},{"auth":"J4FM7FZK","crewName":"Abohasan","vehicle":"GW007","flags":[]},{"auth":"MHQLD2ML","crewName":"Izmarai / Tan","vehicle":"M20","flags":[]},{"auth":"2UNKNIQV","crewName":"Ceja","vehicle":"GW002","flags":[]},{"auth":"UQHWP3D6","crewName":"Goforth","vehicle":"GW003","flags":[]},{"auth":"NBOTXSNC","crewName":"Sapp","vehicle":"WC004","flags":[]}],"sampleUnassigned":[{"auth":"DHW7JF4J","flags":["CREW CONFLICT: overlapping trip"]},{"auth":"GZ45LSKA","flags":["CREW CONFLICT: overlapping trip"]},{"auth":"GO7ZHFLP","flags":["CREW CONFLICT: overlapping trip"]},{"auth"

In [20]:
print(ctx.eval('JSON.stringify(tripsData.slice(0,5))'))
print(ctx.eval('JSON.stringify(Object.values(mappings)[0].results.slice(0,5).map(r=>({auth:r.auth,los:r.los,zone:r.zone,isWR:r.isWR,status:r.status,flags:r.flags,pickup:r.pickup})) )'))


[{"auth":"YBNUNF6W","zone":"SACRAMENTO","los":"WHEELCHAIR","pickup":"06:30","pickupMins":390,"date":"2026-04-17","isWR":false,"specialEquip":"","patient":"NADINE CURTIS","patientDob":"1950-10-31","patientId":"NADINE CURTIS|1950-10-31","travelMins":33,"pickupAddr":"9332 WAYNE HEINTZ STREET,   0 Steps, ELK GROVE, CA, 95624","dropoffAddr":"7420 SHELDON RD,  Dialysis 0  Steps, ELK GROVE , CA, 95758","estTransport":null,"isWrMaster":false},{"auth":"JEAOJFM6","zone":"SACRAMENTO","los":"WHEELCHAIR","pickup":"07:30","pickupMins":450,"date":"2026-04-17","isWR":false,"specialEquip":"","patient":"SOFIA VENTURA","patientDob":"1935-05-01","patientId":"SOFIA VENTURA|1935-05-01","travelMins":11,"pickupAddr":"4785 BUCKNELL COURT, Home 0 Steps, SACRAMENTO, CA, 95841","dropoffAddr":"7000 STOCKTON BOULEVARD,  Dialysis 0  Steps, SACRAMENTO, CA, 95823","estTransport":null,"isWrMaster":false},{"auth":"2Z7XDFPB","zone":"SACRAMENTO","los":"WHEELCHAIR","pickup":"08:15","pickupMins":495,"date":"2026-04-17","isW

In [21]:
print(ctx.eval('JSON.stringify(tripsData[0])'))
print(ctx.eval('JSON.stringify(Object.values(mappings)[0].results[0])'))


{"auth":"YBNUNF6W","zone":"SACRAMENTO","los":"WHEELCHAIR","pickup":"06:30","pickupMins":390,"date":"2026-04-17","isWR":false,"specialEquip":"","patient":"NADINE CURTIS","patientDob":"1950-10-31","patientId":"NADINE CURTIS|1950-10-31","travelMins":33,"pickupAddr":"9332 WAYNE HEINTZ STREET,   0 Steps, ELK GROVE, CA, 95624","dropoffAddr":"7420 SHELDON RD,  Dialysis 0  Steps, ELK GROVE , CA, 95758","estTransport":null,"isWrMaster":false}
{"auth":"YBNUNF6W","zone":"SACRAMENTO","los":"WHEELCHAIR","pickup":"06:30","pickupMins":390,"date":"2026-04-17","isWR":false,"specialEquip":"","patient":"NADINE CURTIS","patientDob":"1950-10-31","patientId":"NADINE CURTIS|1950-10-31","travelMins":33,"pickupAddr":"9332 WAYNE HEINTZ STREET,   0 Steps, ELK GROVE, CA, 95624","dropoffAddr":"7420 SHELDON RD,  Dialysis 0  Steps, ELK GROVE , CA, 95758","estTransport":null,"crew":[],"crewName":null,"vehicle":null,"status":"Unassigned","flags":["WAIT/RETURN NO SAVED CREW FOR PATIENT","No EMT fallback for wheelchair 

In [25]:
print('roster count', ctx.eval('roster.length'))
print('first roster', ctx.eval('JSON.stringify(roster.slice(0,10))'))
print('active non-emts in SACRAMENTO', ctx.eval("JSON.stringify(roster.filter(e=>e.type==='Non-EMT'&&e.zone==='SACRAMENTO').slice(0,10))"))
print('mapping empState count', ctx.eval('JSON.stringify(Object.values(mappings["04/17/2026"].empState).length)'))


roster count 30
first roster [{"id":129,"name":"Muzill Izmarai","zone":"SACRAMENTO","type":"EMT","status":"Active","days":[1,1,1,1,1,1,1]},{"id":130,"name":"Ghazi Abed","zone":"SACRAMENTO","type":"EMT","status":"Active","days":[0,0,1,1,0,1,1]},{"id":131,"name":"Evan  Seidell","zone":"SACRAMENTO","type":"EMT","status":"Active","days":[1,1,1,1,1,1,0]},{"id":132,"name":"Meontae Sapp","zone":"SANTA CLARA","type":"EMT","status":"Active","days":[1,1,1,1,1,1,0]},{"id":133,"name":"William Rodriguez","zone":"SANTA CLARA","type":"EMT","status":"Active","days":[1,1,1,1,1,1,0]},{"id":134,"name":"Cole Flores","zone":"SANTA CLARA","type":"EMT","status":"Active","days":[1,0,1,1,1,1,0]},{"id":135,"name":"Alex Tan","zone":"SACRAMENTO","type":"EMT","status":"Active","days":[1,1,1,1,1,1,1]},{"id":136,"name":"Jose Ceja","zone":"SACRAMENTO","type":"EMT","status":"Active","days":[1,1,1,1,1,1,1]},{"id":137,"name":"Ed Goforth","zone":"SACRAMENTO","type":"EMT","status":"Active","days":[1,1,1,1,1,1,1]},{"id":13

JSEvalException: Uncaught TypeError: Cannot read property 'empState' of undefined at undefined:1:52
TypeError: Cannot read property 'empState' of undefined
    at <anonymous>:1:53

In [28]:
print(ctx.eval('JSON.stringify(usersRaw[2])'))
print(ctx.eval('JSON.stringify(usersRaw[2]["User Group"])'))


{"Name":"Evan  Seidell","User Group":"Non-EMT","STATUS":"ACTIVE","Location Setup":"SACRAMENTO","MON":"YES","TUE":"YES","WED":"YES","THUR":"YES","FRI":"YES","SAT":"YES","SUN":"NO","":""}
"Non-EMT"


In [29]:
print(ctx.eval('normalizeCrewType("Non-EMT")'))
print(ctx.eval('normalizeCrewType("EMT")'))


Non-EMT
EMT


In [30]:
print('roster[2].type', ctx.eval('roster[2].type'))
print('roster[2].name', ctx.eval('roster[2].name'))
print('roster[2].zone', ctx.eval('roster[2].zone'))


roster[2].type Non-EMT
roster[2].name Evan  Seidell
roster[2].zone SACRAMENTO


In [31]:
print('vehicle count', ctx.eval('vehicles.length'))
print(ctx.eval('JSON.stringify(vehicles.slice(0,10))'))


vehicle count 17
[{"id":300,"name":"GW011","plate":"","type":"Gurney Van","status":"Active","location":"SANTA CLARA","bariatric":true},{"id":301,"name":"GW012","plate":"","type":"Gurney Van","status":"Active","location":"SACRAMENTO","bariatric":true},{"id":302,"name":"GW010","plate":"","type":"Gurney Van","status":"Active","location":"SACRAMENTO","bariatric":true},{"id":303,"name":"GW009","plate":"","type":"Gurney Van","status":"Active","location":"SACRAMENTO","bariatric":true},{"id":304,"name":"GW008","plate":"","type":"Gurney Van","status":"Active","location":"SANTA CLARA","bariatric":true},{"id":305,"name":"GW007","plate":"","type":"Gurney Van","status":"Active","location":"SACRAMENTO","bariatric":true},{"id":306,"name":"WC005","plate":"","type":"Wheelchair Van","status":"Active","location":"SACRAMENTO","bariatric":false},{"id":307,"name":"WC004","plate":"","type":"Wheelchair Van","status":"Active","location":"SANTA CLARA","bariatric":false},{"id":308,"name":"M26","plate":"","type":

In [32]:
print(ctx.eval('JSON.stringify(vehiclesRaw[0])'))
print(ctx.eval('get(vehiclesRaw[0], "Bariatric Capable", "Bariatric", "bariatric")'))
print(ctx.eval('get(vehiclesRaw[0], "Vehicle Type", "Type", "type")'))
print(ctx.eval('get(vehiclesRaw[0], "Location", "Zone", "Base", "location")'))


{"Vehicle Name":"GW011","Status":"Active","Zone":"Santa Clara","Certification":"Non-EMT","Bariatric":"TRUE"}
TRUE

Santa Clara


In [ ]:
import json

res_json = ctx.eval("JSON.stringify(mappings['04/17/2026'].results)")
res = json.loads(res_json)
assigned = [r for r in res if r['status']=='Assigned']
unassigned = [r for r in res if r['status']=='Unassigned']
partial = [r for r in res if r['status']=='Partial']
blocked = [r for r in res if r['status']=='Blocked']

print(f"Total trips: {len(res)}")
print(f"Assigned: {len(assigned)}")
print(f"Unassigned: {len(unassigned)}")
print(f"Partial: {len(partial)}")
print(f"Blocked: {len(blocked)}")

print('\nAssigned trip details (first 20):')
for r in assigned[:20]:
    print(f"{r['auth']} | {r['zone']} | {r['los']} | {r['pickup']} | crew={r.get('crewName')} | vehicle={r.get('vehicle')} | flags={r.get('flags')}")

print('\nUnassigned trip details (first 20):')
for r in unassigned[:20]:
    print(f"{r['auth']} | {r['zone']} | {r['los']} | {r['pickup']} | flags={r.get('flags')}")

# Reason summary
reason_counts = {}
for r in unassigned:
    for f in r.get('flags', []):
        reason_counts[f] = reason_counts.get(f, 0) + 1
print('\nUnassigned reason counts:')
for reason, count in sorted(reason_counts.items(), key=lambda x: -x[1]):
    print(f"{count} x {reason}")
